# TensorStream — throughput & GPU-utilization benchmark

Runs on a **Colab GPU runtime** (`Runtime → Change runtime type → GPU`).

It measures, over a steady-state window (startup / drain excluded), for both
the real 3-process pipeline and a naive single-threaded baseline:

* **sustained FPS** = frames / wall-clock seconds
* **mean GPU util %** = average of `nvidia-smi utilization.gpu` samples over the same window

Then prints one summary line: FPS + model + resolution + pipelined GPU util % + single-threaded GPU util %.

In [ ]:
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), 'No GPU — set Runtime type to GPU'
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
# Get the code. Push benchmark.py + make_sample_video.py to the repo first,
# or upload them into the tensorstream/ folder after cloning.
![ -d tensorstream ] || git clone https://github.com/roygalCS/tensorstream.git
%cd tensorstream
!pip -q install opencv-python-headless
import os; assert os.path.exists('benchmark.py'), 'benchmark.py missing — push it to the repo or upload it here'

In [ ]:
# Synthetic 720p clip so there is no download dependency.
# --loops in the next cell replays it, so this stays small.
!python make_sample_video.py --output sample_video.mp4 --width 1280 --height 720 --num-frames 4000

In [ ]:
# Both modes. Bump --loops if the steady-state window prints as too short
# (a fast GPU chews through 4000 frames quickly).
!python benchmark.py --video sample_video.mp4 --mode both \
    --model resnet18 --batch-size 4 --input-size 224 \
    --loops 6 --warmup 2 --cooldown 1

## Knobs

| flag | meaning |
|---|---|
| `--mode` | `pipelined`, `baseline`, or `both` |
| `--model` | `resnet18` / `resnet50` / `mobilenet_v2` |
| `--batch-size` | frames per inference batch |
| `--input-size` | inference resolution (frames are resized to NxN) |
| `--loops` | replay the clip N times for a longer sustained run |
| `--warmup` / `--cooldown` | seconds of startup / drain excluded from FPS + util |

`resnet50` makes the run more GPU-bound and widens the pipelined-vs-baseline gap.